In [8]:
import pandas as pd
import numpy as np
ctd = pd.read_csv("../Data/downcast_all.csv")
print(ctd.head())

/var/folders/xn/9gqx3sxx3s32k1ttvhjc4fch0000gn/T/ipykernel_77701/975230851.py:3: DtypeWarning: Columns (2,3,7,8,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  ctd = pd.read_csv("../Data/downcast_all.csv")


   PROJECT   STUDY ORD_OCC EVENT_NUM    CAST_ID         DATE_TIME_UTC  \
0  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:05:46Z   
1  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:04Z   
2  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:10Z   
3  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:11Z   
4  CalCOFI  9308NH       1       NaN  9308_001d  1993-08-11T12:06:14Z   

          DATE_TIME_PST LAT_DEC LON_DEC       STA_ID  ...  OXBUM CHL_A PHAEO  \
0  1993-08-11T04:05:46Z     NaN     NaN  093.3 026.7  ...    NaN   NaN   NaN   
1  1993-08-11T04:06:04Z   -99.0   -99.0  093.3 026.7  ...  242.7  0.18  0.05   
2  1993-08-11T04:06:10Z   -99.0   -99.0  093.3 026.7  ...    NaN   NaN   NaN   
3  1993-08-11T04:06:11Z   -99.0   -99.0  093.3 026.7  ...  243.9  0.16  0.05   
4  1993-08-11T04:06:14Z   -99.0   -99.0  093.3 026.7  ...    NaN   NaN   NaN   

   NO3  NO2  NH4  PO4  SIL FINAL_FLAG CAST_COUNT  
0  NaN  NaN  NaN  NaN  NaN   

In [ ]:
ctd_out = ctd.drop(columns=["ORD_OCC", "EVENT_NUM"])

# Coerce all mixed-type object columns to numeric where possible
for col in ctd_out.select_dtypes(include="object").columns:
    converted = pd.to_numeric(ctd_out[col], errors="coerce")
    if converted.notna().sum() > 0:  # only replace if column has numeric data
        ctd_out[col] = converted

ctd_out.to_parquet("../Data/Parquet/downcast_all.parquet", index=False)
print("Saved to ../Data/Parquet/downcast_all.parquet")

Saved to ../Data/Parquet/downcast_all.parquet


: 

In [1]:
import pandas as pd
secchi = pd.read_csv("../Data/1949-2021_Secchi.csv", low_memory=False)

for col in secchi.select_dtypes(include="object").columns:
    converted = pd.to_numeric(secchi[col], errors="coerce")
    if converted.notna().sum() > 0:
        secchi[col] = converted

secchi.to_parquet("../Data/Parquet/1949-2021_Secchi.parquet", index=False)
print("Saved to ../Data/Parquet/1949-2021_Secchi.parquet")

Saved to ../Data/Parquet/1949-2021_Secchi.parquet


## Run the following only once to merge datasets

In [2]:
downcast = pd.read_parquet("../Data/Parquet/downcast_all.parquet")
secchi = pd.read_parquet("../Data/Parquet/1949-2021_Secchi.parquet")

secchi = secchi.rename(columns={"Cst_Cnt": "CAST_COUNT"})

merged = downcast.merge(secchi, on="CAST_COUNT", how="inner")

merged.to_parquet("../Data/Parquet/downcast_secchi_merged.parquet", index=False)
print(f"Merged shape: {merged.shape}")
print("Saved to ../Data/Parquet/downcast_secchi_merged.parquet")

Merged shape: (3545372, 142)
Saved to ../Data/Parquet/downcast_secchi_merged.parquet


In [3]:
print(merged["Secchi"].isna().sum())

2141218


## Run code from here for each session

In [3]:
import pandas as pd

merged = pd.read_parquet("../Data/Parquet/downcast_secchi_merged.parquet")
valid = merged[(merged["LAT_DEC"] != -99.0) & (merged["LON_DEC"] != -99.0)]
print(valid["CAST_COUNT"].nunique())

8072


### Format of this merged dataset
Need to aggregate the CTD variables so that there is 1 value per cast count and thus one response (Secchi value) too
- Create the photic depth threshold again - 2m baseline with 4.7% light threshold
- Aggregate XMiss, CHL-A, ESTCHL, PHAEO, (sum to min{photic_depth, median_secchi=20}), calc KPAR and KPAR_SLOPE 
- Select cast counts to predict secchi for times within the 1 hr after sunrise 1 hour before sunset interval, by using real sunset/sunrise times for the specific date_time

In [4]:
# Filter NaN values in LAT_DEC and LON_DEC
merged_filtered = merged.copy()
merged_filtered[["LAT_DEC", "LON_DEC"]] = merged_filtered[["LAT_DEC", "LON_DEC"]].replace(-99.0, float("nan"))
print(merged_filtered[["LAT_DEC", "LON_DEC"]].isna().sum()) # print missing values

LAT_DEC    629326
LON_DEC    631063
dtype: int64


Note: merged_filtered, the df with -99.0 coords replaced with NaN is not written to parquet yet

In [5]:
unique_coords = merged_filtered[["LAT_DEC", "LON_DEC"]].drop_duplicates().dropna()
print(f"Unique (LAT_DEC, LON_DEC) pairs: {len(unique_coords)}")
print(unique_coords.to_string())

Unique (LAT_DEC, LON_DEC) pairs: 45119
              LAT_DEC       LON_DEC
420588   3.508000e+01 -1.207800e+02
629387   3.294000e+01 -1.173000e+02
629452   3.291000e+01 -1.173900e+02
629971   3.284000e+01 -1.175300e+02
630491   3.268000e+01 -1.178700e+02
631003   3.251000e+01 -1.182100e+02
631524   3.234000e+01 -1.185500e+02
632564   3.218000e+01 -1.189200e+02
633075   3.201000e+01 -1.192200e+02
633586   3.184000e+01 -1.195700e+02
634103   3.151000e+01 -1.202400e+02
634616   3.117000e+01 -1.209400e+02
635130   3.084000e+01 -1.215800e+02
635647   3.051000e+01 -1.222500e+02
636166   3.018000e+01 -1.229200e+02
636678   2.985000e+01 -1.236000e+02
637191   3.041000e+01 -1.239900e+02
637706   3.075000e+01 -1.233300e+02
638227   3.111000e+01 -1.226700e+02
638747   3.141000e+01 -1.219900e+02
639267   3.175000e+01 -1.213100e+02
639791   3.208000e+01 -1.206400e+02
640309   3.241000e+01 -1.199500e+02
640831   3.264000e+01 -1.194800e+02
641351   3.292000e+01 -1.189300e+02
641873   3.318000e+01 -1.

In [6]:
# Remove CAST_COUNTs that have any invalid (NaN) coordinates
invalid_cast_counts = set(
    merged_filtered[merged_filtered["LAT_DEC"].isna() | merged_filtered["LON_DEC"].isna()]["CAST_COUNT"].unique()
)
merged_filtered = merged_filtered[~merged_filtered["CAST_COUNT"].isin(invalid_cast_counts)]
print(f"Remaining CAST_COUNTs after removing those with invalid coords: {merged_filtered['CAST_COUNT'].nunique()}")
print(f"Remaining rows: {len(merged_filtered)}")

Remaining CAST_COUNTs after removing those with invalid coords: 6625
Remaining rows: 2888074


The Latitude and Longitude coordinates of each station are very similar, and the CCE is a relatively small region likely to be unaffected by taking an average of coordinates to use to calculate sunrise/sunset times.

Thus, I'll use a single average of each coordinate.

In [7]:
# Average across all rows with valid coordinates
avg_lat = merged_filtered["LAT_DEC"].mean()
avg_lon = merged_filtered["LON_DEC"].mean()
print(f"Average LAT: {avg_lat:.4f}")
print(f"Average LON: {avg_lon:.4f}")

Average LAT: 32.8368
Average LON: -120.5390


So we'll use the average `LAT` and `LON` along with `DATE_TIME_UTC` to find the daylight window for each cast.

In [71]:
print(merged_filtered.head())

        PROJECT   STUDY    CAST_ID         DATE_TIME_UTC  \
420588  CalCOFI  9908NH  9908_067d  1997-01-01T01:12:39Z   
420589  CalCOFI  9908NH  9908_067d  1997-01-01T01:12:51Z   
420590  CalCOFI  9908NH  9908_067d  1997-01-01T01:12:54Z   
420591  CalCOFI  9908NH  9908_067d  1997-01-01T01:12:56Z   
420592  CalCOFI  9908NH  9908_067d  1997-01-01T01:12:58Z   

               DATE_TIME_PST  LAT_DEC  LON_DEC       STA_ID  LINE   STA  ...  \
420588  1996-12-31T17:12:39Z    35.08  -120.78  076.7 049.0  76.7  49.0  ...   
420589  1996-12-31T17:12:51Z    35.08  -120.78  076.7 049.0  76.7  49.0  ...   
420590  1996-12-31T17:12:54Z    35.08  -120.78  076.7 049.0  76.7  49.0  ...   
420591  1996-12-31T17:12:56Z    35.08  -120.78  076.7 049.0  76.7  49.0  ...   
420592  1996-12-31T17:12:58Z    35.08  -120.78  076.7 049.0  76.7  49.0  ...   

        Wave_Prd  Wind_Dir  Wind_Spd  Barometer  Dry_T  Wet_T  Wea  Cloud_Typ  \
420588       7.0      28.0       6.0     1013.2   13.8   12.8  4.0        NaN

In [4]:
!pip install astral -q

In [8]:
from astral import LocationInfo
from astral.sun import sun
from datetime import timezone, timedelta
from zoneinfo import ZoneInfo

LA_TZ = ZoneInfo("America/Los_Angeles")

df = merged_filtered[["CAST_COUNT", "DATE_TIME_UTC"]].copy()
df["DATE_TIME_UTC"] = pd.to_datetime(df["DATE_TIME_UTC"], utc=True)
df = df.dropna(subset=["DATE_TIME_UTC"])

# Extract local date using America/Los_Angeles (accounts for DST automatically)
df["local_date"] = df["DATE_TIME_UTC"].dt.tz_convert(LA_TZ).dt.date

# Single location using average coordinates
loc = LocationInfo(latitude=avg_lat, longitude=avg_lon)

# Compute sunrise/sunset once per unique local date using LA timezone
date_cache = {}
for d in df["local_date"].unique():
    try:
        s = sun(loc.observer, date=d, tzinfo=LA_TZ)
        date_cache[d] = (s["sunrise"] + timedelta(hours=1), s["sunset"] - timedelta(hours=1))
    except Exception:
        date_cache[d] = (None, None)

df["window_start"] = df["local_date"].map(lambda d: date_cache.get(d, (None, None))[0])
df["window_end"]   = df["local_date"].map(lambda d: date_cache.get(d, (None, None))[1])

mask = (
    df["window_start"].notna() &
    df["window_end"].notna() &
    (df["DATE_TIME_UTC"] >= df["window_start"]) &
    (df["DATE_TIME_UTC"] <= df["window_end"])
)

daylight_cast_counts = df.loc[mask, "CAST_COUNT"].unique()
daylight_cast_set = set(daylight_cast_counts)
print(f"CAST_COUNT values within daylight window: {len(daylight_cast_counts)}")
print(daylight_cast_counts[:20]) # print tail of cast_counts in daylight window

CAST_COUNT values within daylight window: 2975
[28867. 28868. 28869. 28872. 28873. 28874. 28876. 28877. 28880. 28881.
 28883. 28884. 28887. 28888. 28892. 28893. 28894. 28898. 28899. 28902.]


In [9]:
from astral import LocationInfo
from astral.sun import sun
from datetime import timezone, timedelta, date as date_type
from zoneinfo import ZoneInfo

LA_TZ = ZoneInfo("America/Los_Angeles")
loc_test = LocationInfo(latitude=avg_lat, longitude=avg_lon)
s_test = sun(loc_test.observer, date=date_type(2000, 8, 11), tzinfo=LA_TZ)
window_start = s_test["sunrise"] + timedelta(hours=1)
window_end   = s_test["sunset"]  - timedelta(hours=1)
print(f"Sunrise+1h (UTC):   {window_start.astimezone(timezone.utc)}")
print(f"Sunset-1h  (UTC):   {window_end.astimezone(timezone.utc)}")
print(f"Sunrise+1h (local): {window_start}")
print(f"Sunset-1h  (local): {window_end}")
print(f"Window valid (start < end): {window_start < window_end}")

Sunrise+1h (UTC):   2000-08-11 14:23:16.618283+00:00
Sunset-1h  (UTC):   2000-08-12 01:50:41.837429+00:00
Sunrise+1h (local): 2000-08-11 07:23:16.618283-07:00
Sunset-1h  (local): 2000-08-11 18:50:41.837429-07:00
Window valid (start < end): True


In [10]:
# Rows with invalid coordinates (NaN) - should be 0 after filtering
invalid_coord_mask = (
    merged_filtered["LAT_DEC"].isna() |
    merged_filtered["LON_DEC"].isna()
)

invalid_coords_df = merged_filtered[invalid_coord_mask]

# Of those, which CAST_COUNTs were identified as within the daylight window
# (inferred from other rows of the same cast with valid coordinates)
daylight_cast_set = set(daylight_cast_counts)
inferred_daylight = invalid_coords_df[invalid_coords_df["CAST_COUNT"].isin(daylight_cast_set)]

inferred_cast_counts = inferred_daylight["CAST_COUNT"].unique()
print(f"CAST_COUNT values with invalid coords but inferred to be in daylight window: {len(inferred_cast_counts)}")
print(inferred_cast_counts[:20])

CAST_COUNT values with invalid coords but inferred to be in daylight window: 0
[]


### Keep df with necessary 10 columns only, before feature engineering

In [11]:
cols = ["CAST_ID", "DATE_TIME_UTC", "DATE_TIME_PST", "LAT_DEC", "LON_DEC", "DEPTH", "ESTCHL_STACORR", "XMISS", "PAR", "CAST_COUNT", "Secchi"]
df_slim = merged_filtered[merged_filtered["CAST_COUNT"].isin(daylight_cast_set)][cols].copy()
print(f"Shape: {df_slim.shape}")
print(df_slim.head())

Shape: (1290287, 11)
          CAST_ID         DATE_TIME_UTC         DATE_TIME_PST  LAT_DEC  \
629971  9807_003d  1998-07-08T15:40:47Z  1998-07-08T07:40:47Z    32.84   
629972  9807_003d  1998-07-08T15:41:02Z  1998-07-08T07:41:02Z    32.84   
629973  9807_003d  1998-07-08T15:41:06Z  1998-07-08T07:41:06Z    32.84   
629974  9807_003d  1998-07-08T15:41:09Z  1998-07-08T07:41:09Z    32.84   
629975  9807_003d  1998-07-08T15:41:12Z  1998-07-08T07:41:12Z    32.84   

        LON_DEC  DEPTH  ESTCHL_STACORR  XMISS   PAR  CAST_COUNT  Secchi  
629971  -117.53    2.0            0.06  44.08  0.64     28867.0    32.0  
629972  -117.53    3.0            0.06  44.08  0.64     28867.0    32.0  
629973  -117.53    4.0            0.05  44.09  0.64     28867.0    32.0  
629974  -117.53    5.0            0.05  44.08  0.64     28867.0    32.0  
629975  -117.53    6.0            0.05  44.09  0.64     28867.0    32.0  


In [76]:
# Minimum depth per cast (i.e. the shallowest measurement = starting depth)
min_depth_per_cast = df_slim.groupby("CAST_COUNT")["DEPTH"].min()

# Round to nearest integer and count how many casts start at each depth
starting_depth_counts = min_depth_per_cast.round(0).astype(int).value_counts().sort_index()
print("Starting depth (m) : # of CAST_COUNTs")
print(starting_depth_counts.to_string())

Starting depth (m) : # of CAST_COUNTs
DEPTH
1       814
2      1655
3       428
4        57
5         5
6         4
7         1
8         2
11        1
12        5
13        1
883       1
885       1


In [12]:
casts_with_2m = df_slim[df_slim["DEPTH"].round(0) == 2]["CAST_COUNT"].nunique()
print(f"Unique CAST_COUNTs with a 2m depth measurement: {casts_with_2m}")

Unique CAST_COUNTs with a 2m depth measurement: 2469


In [13]:
non_integer_depths = df_slim[df_slim["DEPTH"] != df_slim["DEPTH"].round(0)]
print(f"Rows with non-integer DEPTH: {len(non_integer_depths)}")
print(non_integer_depths["DEPTH"].describe())
print(non_integer_depths["DEPTH"].head(20))

Rows with non-integer DEPTH: 0
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: DEPTH, dtype: float64
Series([], Name: DEPTH, dtype: float64)


In [14]:
# Only keep CAST_COUNTs that have a measurement at exactly 2m
casts_with_2m = df_slim[df_slim["DEPTH"] == 2]["CAST_COUNT"].unique()
df_photic = df_slim[df_slim["CAST_COUNT"].isin(casts_with_2m)].copy()

# Get PAR at 2m baseline for each cast
par_at_2m = (
    df_photic[df_photic["DEPTH"] == 2]
    .groupby("CAST_COUNT")["PAR"]
    .first()
    .rename("PAR_2m")
)
df_photic = df_photic.merge(par_at_2m, on="CAST_COUNT", how="left")

# Calculate photic zone threshold (4.7% of PAR at 2m)
df_photic["PAR_THRESHOLD"] = df_photic["PAR_2m"] * 0.047

# For each cast, find the row whose PAR is closest to the threshold
df_photic["PAR_DIFF"] = (df_photic["PAR"] - df_photic["PAR_THRESHOLD"]).abs()
photic_idx = df_photic.groupby("CAST_COUNT")["PAR_DIFF"].idxmin().dropna()

df_photic["PHOTIC_ZONE"] = False
df_photic.loc[photic_idx, "PHOTIC_ZONE"] = True

print(f"CAST_COUNTs with photic zone identified: {df_photic["PHOTIC_ZONE"].sum()}")
print(df_photic[df_photic["PHOTIC_ZONE"]][["CAST_COUNT", "DEPTH", "PAR", "PAR_2m", "PAR_THRESHOLD"]].head(10))

CAST_COUNTs with photic zone identified: 2219
      CAST_COUNT  DEPTH   PAR  PAR_2m  PAR_THRESHOLD
0        28867.0    2.0  0.64    0.64        0.03008
520      28868.0    1.0  0.64    0.64        0.03008
1032     28869.0    1.0  0.64    0.64        0.03008
1553     28872.0    1.0  0.64    0.64        0.03008
2064     28874.0    1.0  0.64    0.64        0.03008
2577     28877.0    2.0  0.64    0.64        0.03008
3096     28880.0    2.0  0.64    0.64        0.03008
3611     28881.0    2.0  0.64    0.64        0.03008
4132     28883.0    2.0  0.64    0.64        0.03008
4652     28884.0    2.0  0.64    0.64        0.03008


In [15]:
missing_secchi = (
    df_slim.groupby("CAST_COUNT")[["Secchi", "DATE_TIME_UTC"]]
    .first()
    .reset_index()
)
missing_secchi = missing_secchi[missing_secchi["Secchi"].isna()][["CAST_COUNT", "DATE_TIME_UTC"]]
print(f"Unique CAST_COUNTs with missing Secchi: {len(missing_secchi)}")
print(missing_secchi.to_string())

Unique CAST_COUNTs with missing Secchi: 970
      CAST_COUNT         DATE_TIME_UTC
1        28868.0  1998-07-08T20:00:46Z
2        28869.0  1998-07-09T00:12:10Z
4        28873.0  1998-07-09T17:50:18Z
5        28874.0  1998-07-10T00:06:17Z
7        28877.0  1998-07-10T20:51:42Z
9        28881.0  1998-07-11T23:21:02Z
11       28884.0  1998-07-12T22:12:59Z
12       28887.0  1998-07-14T16:06:45Z
13       28888.0  1998-07-14T22:18:47Z
15       28893.0  1998-07-15T22:32:59Z
16       28894.0  1998-07-16T00:18:40Z
17       28898.0  1998-07-16T17:42:44Z
18       28899.0  1998-07-16T22:01:35Z
19       28902.0  1998-07-17T17:17:59Z
20       28903.0  1998-07-18T00:27:17Z
22       28906.0  1998-07-18T22:21:15Z
23       28909.0  1998-07-19T18:25:28Z
24       28910.0  1998-07-20T00:57:34Z
27       28915.0  1998-07-20T22:59:19Z
28       28919.0  1998-07-21T16:35:40Z
29       28920.0  1998-07-21T23:06:48Z
30       28923.0  1998-07-22T17:24:09Z
31       28924.0  1998-07-22T23:13:07Z
32       28927.0  19

In [16]:
missing_in_daylight = missing_secchi[missing_secchi["CAST_COUNT"].isin(daylight_cast_set)]
print(f"Missing Secchi CAST_COUNTs within daylight window: {len(missing_in_daylight)}")
print(missing_in_daylight.to_string())

Missing Secchi CAST_COUNTs within daylight window: 970
      CAST_COUNT         DATE_TIME_UTC
1        28868.0  1998-07-08T20:00:46Z
2        28869.0  1998-07-09T00:12:10Z
4        28873.0  1998-07-09T17:50:18Z
5        28874.0  1998-07-10T00:06:17Z
7        28877.0  1998-07-10T20:51:42Z
9        28881.0  1998-07-11T23:21:02Z
11       28884.0  1998-07-12T22:12:59Z
12       28887.0  1998-07-14T16:06:45Z
13       28888.0  1998-07-14T22:18:47Z
15       28893.0  1998-07-15T22:32:59Z
16       28894.0  1998-07-16T00:18:40Z
17       28898.0  1998-07-16T17:42:44Z
18       28899.0  1998-07-16T22:01:35Z
19       28902.0  1998-07-17T17:17:59Z
20       28903.0  1998-07-18T00:27:17Z
22       28906.0  1998-07-18T22:21:15Z
23       28909.0  1998-07-19T18:25:28Z
24       28910.0  1998-07-20T00:57:34Z
27       28915.0  1998-07-20T22:59:19Z
28       28919.0  1998-07-21T16:35:40Z
29       28920.0  1998-07-21T23:06:48Z
30       28923.0  1998-07-22T17:24:09Z
31       28924.0  1998-07-22T23:13:07Z
32       

#### All the missing Secchi values are `CAST_COUNT`'s that are within the daylight window

### Just keep rows that have missing Secchi values within the daylight window

In [17]:
target_casts = set(missing_in_daylight["CAST_COUNT"])
df_slim = df_slim[df_slim["CAST_COUNT"].isin(target_casts)]
print(f"Remaining CAST_COUNTs: {df_slim['CAST_COUNT'].nunique()}")
print(f"Remaining rows: {len(df_slim)}")

Remaining CAST_COUNTs: 970
Remaining rows: 371582


### `K_PAR` and `K_PAR_SLOPE` calculations (and NaNs check)

In [18]:
import numpy as np
from scipy.stats import linregress

# Get photic zone depth per cast (row where PHOTIC_ZONE == True)
photic_rows = df_photic[df_photic["PHOTIC_ZONE"]][["CAST_COUNT", "DEPTH", "PAR"]].rename(
    columns={"DEPTH": "PHOTIC_DEPTH", "PAR": "PAR_z"}
)

# Merge PAR_2m into photic_rows
photic_rows = photic_rows.merge(
    df_photic[df_photic["DEPTH"] == 2].groupby("CAST_COUNT")["PAR"].first().rename("PAR_2m"),
    on="CAST_COUNT", how="left"
)

# K_PAR = -ln(PAR(z) / PAR(2m)) / z
photic_rows["K_PAR"] = -np.log(photic_rows["PAR_z"] / photic_rows["PAR_2m"]) / photic_rows["PHOTIC_DEPTH"]

# K_PAR_SLOPE: linear fit of ln(PAR) vs depth over [2m, min(photic_depth, 20m)]
def calc_kpar_slope(cast_count, photic_depth):
    window_max = min(photic_depth, 20)
    rows = df_photic[
        (df_photic["CAST_COUNT"] == cast_count) &
        (df_photic["DEPTH"] >= 2) &
        (df_photic["DEPTH"] <= window_max) &
        (df_photic["PAR"] > 0)
    ][["DEPTH", "PAR"]].dropna()
    if len(rows) < 2 or rows["DEPTH"].nunique() < 2:
        return np.nan
    slope, _, _, _, _ = linregress(rows["DEPTH"], np.log(rows["PAR"]))
    return -slope

photic_rows["K_PAR_SLOPE"] = photic_rows.apply(
    lambda r: calc_kpar_slope(r["CAST_COUNT"], r["PHOTIC_DEPTH"]), axis=1
)

print(photic_rows[["CAST_COUNT", "PHOTIC_DEPTH", "PAR_2m", "PAR_z", "K_PAR", "K_PAR_SLOPE"]].head(10))
print(f"K_PAR NaNs: " + str(photic_rows["K_PAR"].isna().sum()))
print(f"K_PAR_SLOPE NaNs: " + str(photic_rows["K_PAR_SLOPE"].isna().sum()))

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


   CAST_COUNT  PHOTIC_DEPTH  PAR_2m  PAR_z  K_PAR  K_PAR_SLOPE
0     28867.0           2.0    0.64   0.64   -0.0          NaN
1     28868.0           1.0    0.64   0.64   -0.0          NaN
2     28869.0           1.0    0.64   0.64   -0.0          NaN
3     28872.0           1.0    0.64   0.64   -0.0          NaN
4     28874.0           1.0    0.64   0.64   -0.0          NaN
5     28877.0           2.0    0.64   0.64   -0.0          NaN
6     28880.0           2.0    0.64   0.64   -0.0          NaN
7     28881.0           2.0    0.64   0.64   -0.0          NaN
8     28883.0           2.0    0.64   0.64   -0.0          NaN
9     28884.0           2.0    0.64   0.64   -0.0          NaN
K_PAR NaNs: 21
K_PAR_SLOPE NaNs: 418


In [19]:
# --- K_PAR NaN diagnosis ---
kpar_nan = photic_rows[photic_rows["K_PAR"].isna()][["CAST_COUNT", "PHOTIC_DEPTH", "PAR_z", "PAR_2m"]]
print(f"K_PAR NaN rows ({len(kpar_nan)}):", kpar_nan.to_string())

print("--- K_PAR_SLOPE NaN diagnosis ---")
kpar_slope_nan_casts = photic_rows[photic_rows["K_PAR_SLOPE"].isna()]["CAST_COUNT"].values

reasons = {"PAR_missing_or_zero": 0, "fewer_than_2_rows": 0, "all_same_depth": 0}
for cast_count in kpar_slope_nan_casts:
    photic_depth = photic_rows.loc[photic_rows["CAST_COUNT"] == cast_count, "PHOTIC_DEPTH"].values[0]
    window_max = min(photic_depth, 20)
    rows = df_photic[
        (df_photic["CAST_COUNT"] == cast_count) &
        (df_photic["DEPTH"] >= 2) &
        (df_photic["DEPTH"] <= window_max) &
        (df_photic["PAR"] > 0)
    ][["DEPTH", "PAR"]].dropna()
    if len(rows) < 2:
        reasons["fewer_than_2_rows"] += 1
    elif rows["DEPTH"].nunique() < 2:
        reasons["all_same_depth"] += 1
    else:
        reasons["PAR_missing_or_zero"] += 1

print("Reason breakdown:", reasons)

# Sample window for first 3 slope-NaN casts
print("Sample window contents for first 3 K_PAR_SLOPE NaN casts:")
for cast_count in kpar_slope_nan_casts[:3]:
    photic_depth = photic_rows.loc[photic_rows["CAST_COUNT"] == cast_count, "PHOTIC_DEPTH"].values[0]
    window_max = min(photic_depth, 20)
    rows = df_photic[
        (df_photic["CAST_COUNT"] == cast_count) &
        (df_photic["DEPTH"] >= 2) &
        (df_photic["DEPTH"] <= window_max)
    ][["DEPTH", "PAR"]]
    print(f"CAST_COUNT={cast_count}, photic_depth={photic_depth}, window=[2, {window_max}]")
    print(rows.to_string())


K_PAR NaN rows (21):       CAST_COUNT  PHOTIC_DEPTH         PAR_z  PAR_2m
146      29281.0           3.0 -6.200000e+01  351.45
523      30376.0          62.0  8.700000e-01  -13.48
648      30784.0          38.0 -9.990000e-29    2.10
649      30785.0          26.0 -9.990000e-29    2.00
651      30790.0          50.0 -9.990000e-29    2.07
652      30794.0          36.0 -9.990000e-29    1.85
653      30780.0          39.0 -9.990000e-29    1.69
654      30776.0          40.0 -9.990000e-29    1.95
657      30755.0          38.0 -9.990000e-29    2.25
658      30760.0          32.0 -9.990000e-29    2.03
659      30764.0          35.0 -9.990000e-29    1.91
660      30749.0          33.0 -9.990000e-29    1.79
662      30744.0          41.0 -9.990000e-29    1.80
664      30732.0          31.0 -9.990000e-29    1.80
665      30733.0          31.0 -9.990000e-29    1.92
666      30737.0          31.0 -9.990000e-29    2.05
667      30728.0          33.0 -9.990000e-29    1.73
668      30724.0         

In [20]:
total = len(photic_rows)
after_kpar = photic_rows[photic_rows["K_PAR"].notna()]
after_both = after_kpar[after_kpar["K_PAR_SLOPE"].notna()]
print(f"Total CAST_COUNTs: {total}")
print(f"After dropping K_PAR NaNs: {len(after_kpar)} (lost {total - len(after_kpar)})")
print(f"After also dropping K_PAR_SLOPE NaNs: {len(after_both)} (lost {len(after_kpar) - len(after_both)})")
print(f"Final usable CAST_COUNTs: {len(after_both)} / {total} ({100*len(after_both)/total:.1f}%)")


Total CAST_COUNTs: 2219
After dropping K_PAR NaNs: 2198 (lost 21)
After also dropping K_PAR_SLOPE NaNs: 1782 (lost 416)
Final usable CAST_COUNTs: 1782 / 2219 (80.3%)


In [21]:
# Keep only casts with valid K_PAR and K_PAR_SLOPE
photic_rows = photic_rows[photic_rows["K_PAR"].notna() & photic_rows["K_PAR_SLOPE"].notna()].reset_index(drop=True)
print(f"Retained CAST_COUNTs: {len(photic_rows)}")


Retained CAST_COUNTs: 1782


### So, after filtering the K_PAR/slope values for NaNs, we still have 80% of Casts usable (1782)

### Now build integration window to sum up XMISS and ESTCHL

In [22]:
# Build integration window per cast: [2m, min(photic_depth, 20m)]
photic_depth_map = photic_rows.set_index("CAST_COUNT")["PHOTIC_DEPTH"]

def sum_over_window(cast_count, col):
    window_max = min(photic_depth_map[cast_count], 20)
    rows = df_photic[
        (df_photic["CAST_COUNT"] == cast_count) &
        (df_photic["DEPTH"] >= 2) &
        (df_photic["DEPTH"] <= window_max)
    ][col]
    return rows.sum(min_count=1)

cast_counts = photic_rows["CAST_COUNT"].values

photic_rows["XMISS_SUMMED"]  = [sum_over_window(c, "XMISS")          for c in cast_counts]
photic_rows["ESTCHL_SUMMED"] = [sum_over_window(c, "ESTCHL_STACORR") for c in cast_counts]

print(photic_rows[["CAST_COUNT", "XMISS_SUMMED", "ESTCHL_SUMMED"]].head(10))
print("NaN counts:")
print(photic_rows[["XMISS_SUMMED", "ESTCHL_SUMMED"]].isna().sum())


   CAST_COUNT  XMISS_SUMMED  ESTCHL_SUMMED
0     29275.0        798.45           1.71
1     29276.0        801.46           1.53
2     29277.0        795.90           3.08
3     29282.0         86.56           0.18
4     29285.0        809.93           0.87
5     29292.0         84.87           0.12
6     29293.0        802.06           1.71
7     29296.0        776.98           6.58
8     29380.0        798.59           1.37
9     29384.0        787.90          12.96
NaN counts:
XMISS_SUMMED      0
ESTCHL_SUMMED    17
dtype: int64


### `CHL_A_SUMMED` & `PHAEO_SUMMED` are very sparse which is why they weren't great predictors in prior analyses, so 

In [23]:
# Drop the 17 casts with missing ESTCHL_SUMMED
photic_rows = photic_rows[photic_rows["ESTCHL_SUMMED"].notna()].reset_index(drop=True)
print(f"Retained CAST_COUNTs after ESTCHL_SUMMED filter: {len(photic_rows)}")


Retained CAST_COUNTs after ESTCHL_SUMMED filter: 1765


In [25]:
# Pull per-cast metadata from merged_filtered (df_slim was filtered
# to target_casts after df_photic was built, so use merged_filtered instead)
meta_cols = ["CAST_COUNT", "CAST_ID", "DATE_TIME_UTC", "DATE_TIME_PST", "LAT_DEC", "LON_DEC"]
cast_meta = merged_filtered.groupby("CAST_COUNT")[meta_cols[1:]].first().reset_index()

# Build final one-row-per-cast dataset
df_final = (
    photic_rows
    [["CAST_COUNT", "K_PAR", "K_PAR_SLOPE", "PHOTIC_DEPTH", "ESTCHL_SUMMED", "XMISS_SUMMED"]]
    .rename(columns={"PHOTIC_DEPTH": "DEPTH"})
    .merge(cast_meta, on="CAST_COUNT", how="left")
)

# Reorder columns
df_final = df_final[["CAST_COUNT", "CAST_ID", "DATE_TIME_UTC", "DATE_TIME_PST",
                      "LAT_DEC", "LON_DEC", "DEPTH", "K_PAR", "K_PAR_SLOPE",
                      "ESTCHL_SUMMED", "XMISS_SUMMED"]]

print(f"Shape: {df_final.shape}")
print(f"Missing values:{df_final.isna().sum()}")
print(df_final.head())


Shape: (1765, 11)
Missing values:CAST_COUNT       0
CAST_ID          0
DATE_TIME_UTC    0
DATE_TIME_PST    0
LAT_DEC          0
LON_DEC          0
DEPTH            0
K_PAR            0
K_PAR_SLOPE      0
ESTCHL_SUMMED    0
XMISS_SUMMED     0
dtype: int64
   CAST_COUNT    CAST_ID         DATE_TIME_UTC         DATE_TIME_PST  LAT_DEC  \
0     29275.0  9910_003d  1999-10-03T16:51:45Z  1999-10-03T08:51:45Z    32.84   
1     29276.0  9910_004d  1999-10-03T20:49:44Z  1999-10-03T12:49:44Z    32.67   
2     29277.0  9910_005d  1999-10-04T00:42:38Z  1999-10-03T16:42:38Z    32.51   
3     29282.0  9910_010d  1999-10-04T23:56:29Z  1999-10-04T15:56:29Z    31.51   
4     29285.0  9910_013d  1999-10-05T20:59:30Z  1999-10-05T12:59:30Z    30.51   

   LON_DEC  DEPTH     K_PAR  K_PAR_SLOPE  ESTCHL_SUMMED  XMISS_SUMMED  
0  -117.53   54.0  0.010971    -0.000000           1.71        798.45  
1  -117.87   41.0  0.010541    -0.001163           1.53        801.46  
2  -118.20   41.0  0.009938     0.000028  

In [26]:
df_final.to_parquet("../Data/Parquet/daylight_stations_to_predict.parquet", index=False)
print(f"Saved {len(df_final)} rows to ../Data/Parquet/daylight_stations_to_predict.parquet")


Saved 1765 rows to ../Data/Parquet/daylight_stations_to_predict.parquet


## Wrote the final df to parquet: `daylight_stations_to_predict.parquet`